In [21]:
import sys
sys.path.insert(0, './src')

from modeltrainerv2 import ModelML
from dataprocessor import DataProcessor
from func_utils import *

import importlib
import src.modeltrainerv2
importlib.reload(src.modeltrainerv2)

<module 'src.modeltrainerv2' from '/home/mouad-sehli/Desktop/DS final/src/modeltrainerv2.py'>

In [22]:
import pandas as pd

# Read train and test from Excel sheets
train_data = pd.read_excel('data/processed_data_imputed.xlsx', sheet_name='train')
test_data = pd.read_excel('data/processed_data_imputed.xlsx', sheet_name='test')

# Separate features and target
# Drop patient_ids (ID column) from features
X_train = train_data.iloc[:, :-1].drop(columns=['patient_ids','feat_40','feat_82'], errors='ignore')
y_train = train_data.iloc[:, -1]

X_test = test_data.iloc[:, :-1].drop(columns=['patient_ids','feat_40','feat_82'], errors='ignore')
y_test = test_data.iloc[:, -1]

print(f"Train shape: X={X_train.shape}, y={y_train.shape}")
print(f"Test shape: X={X_test.shape}, y={y_test.shape}")

Train shape: X=(83555, 81), y=(83555,)
Test shape: X=(28465, 81), y=(28465,)


In [23]:
X_train

,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,...,feat_72,feat_73,feat_74,feat_75,feat_76,feat_77,feat_78,feat_79,feat_80,feat_81
0,2532.0401,9.0,11.8,166.0,1,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0
1,1062.2320,4.0,11.3,17.0,1,0,0,0,0,0,...,0,1,1,0,0,0,1,0,0,0
2,1115.6200,4.0,10.9,150.0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,852.7050,3.0,6.4,14.0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,1983.6317,11.0,11.0,18.0,1,1,1,0,1,0,...,1,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83550,609.2880,9.0,11.9,119.0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
83551,119.2376,1.0,11.4,28.0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
83552,347.4000,1.0,9.0,310.0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
83553,515.9460,4.0,9.5,115.0,0,1,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0


In [24]:
trainer = ModelML(x_train=X_train, x_test=X_test, y_train=y_train, y_test=y_test)

# RandomizedSearch - plus rapide
trainer.tune_logistic_regression(
    method='random',
    n_iter=100,
    cv=5,
    scoring='roc_auc'
)
trainer.save_model()


🔍 Recherche d'hyperparamètres (RANDOM)
   Scoring: roc_auc
   CV folds: 5
   Paramètres testés: ['C', 'penalty', 'solver', 'class_weight', 'max_iter']
Fitting 5 folds for each of 48 candidates, totalling 240 fits


/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib/pyth


✓ Meilleurs paramètres trouvés:
   - solver: liblinear
   - penalty: l1
   - max_iter: 1000
   - class_weight: balanced
   - C: 1
   - Best CV Score (roc_auc): 0.8051

📊 Résultats:
   - Train Accuracy: 0.7299
   - Test Accuracy: 0.7056
✓ Modèle sauvegardé:  model/LogisticRegression_tuned_20260110_123941.pkl
✓ Paramètres sauvegardés:  model/LogisticRegression_tuned_20260110_123941_params.json


'model/LogisticRegression_tuned_20260110_123941.pkl'

In [28]:
# Après un premier entraînement, ajuster certains params
trainer.update_hyperparameters(C=0.1, class_weight='balanced')
trainer.save_model(filename='updated_model')
trainer = ModelML()
trainer.load_model('model/LogisticRegression_tuned_20260110_123941_params.pkl')
print(trainer.get_hyperparameters())


🔄 Mise à jour des hyperparamètres et ré-entraînement
   Anciens:  None
   Nouveaux:  {'max_iter': 1000, 'random_state': 42, 'C': 0.1, 'class_weight': 'balanced'}


ValueError: Les données d'entraînement ne sont pas définies.